# Практика 2. Точечные и гистограммные преобразования, цвет — версия для студентов

**Курс:** Обработка изображений

На практике мы:
- исследуем распределение яркостей и функции преобразования;
- реализуем растяжение контраста, кусочно-линейное преобразование, логарифм и гамма-коррекцию;
- построим гистограмму, CDF и две версии эквализации без готовой функции;
- сравним глобальную эквализацию и CLAHE и выполним сопоставление гистограмм;
- применим HSV для цветовой сегментации с учётом цикличности Hue;
- исправим цветовой оттенок методом Gray World и проверим границы его применимости;
- реализуем порог Отсу, сравним его с Sauvola и оценим сегментацию по IoU.

Все вычисления выполняем во `float32`; перед возвратом к `uint8` используем `clip`.
Сначала сформулируйте ожидаемый результат, затем запускайте код и объясняйте наблюдение.


## 0. Импорты и вспомогательные функции


In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

from skimage import color, data, exposure, filters, util

print("OpenCV:", cv2.__version__)
print("NumPy:", np.__version__)


In [ ]:
def show(*images, titles=None, cmaps=None, figsize=(15, 5), vmin=None, vmax=None):
    n = len(images)
    titles = titles or [""] * n
    cmaps = cmaps or [None] * n
    fig, axes = plt.subplots(1, n, figsize=figsize, squeeze=False)
    for ax, image, title, cmap in zip(axes[0], images, titles, cmaps):
        ax.imshow(image, cmap=cmap, vmin=vmin, vmax=vmax)
        ax.set_title(title)
        ax.axis("off")
    plt.tight_layout()
    plt.show()


def to_float(image):
    return image.astype(np.float32) / 255.0


gray = data.camera()
rgb = data.astronaut()
show(gray, rgb, titles=["camera: grayscale", "astronaut: RGB"], cmaps=["gray", None])


### 0.1. Диагностика диапазона и распределения

До выбора преобразования полезно проверить тип данных, диапазон, среднее,
стандартное отклонение и крайние перцентили. Стандартное отклонение описывает
разброс яркостей, но не является универсальной мерой качества или контраста.

**Задание 1.** Реализуйте краткий отчёт. Почему одного `min/max` недостаточно?


In [ ]:
def intensity_report(image):
    # TODO: верните словарь с dtype, shape, min, max, mean, std, p01 и p99.
    pass


print(intensity_report(gray))

# Дополнительно: визуализируйте гистограммы gray и low-contrast изображения
# после того, как оно будет создано в следующем разделе.


## 1. Растяжение контраста по перцентилям

Один выброс делает растяжение по `min/max` бесполезным. Поэтому границы полезного
диапазона часто оценивают перцентилями:

$$s = \operatorname{clip}\left(\frac{r-p_1}{p_{99}-p_1}, 0, 1\right).$$


In [ ]:
# Искусственно уменьшим контраст и добавим два выброса.
low_contrast = np.clip(gray.astype(np.float32) * 0.35 + 80, 0, 255).astype(np.uint8)
low_contrast[0, 0] = 0
low_contrast[0, 1] = 255

# TODO:
# 1. Найдите перцентили p1 и p99.
# 2. Перейдите во float32.
# 3. Растяните диапазон [p1, p99] в [0, 1] и примените clip.
p1, p99 = ...
stretched = ...

print(f"p1={p1:.1f}, p99={p99:.1f}")
show(
    to_float(low_contrast), stretched,
    titles=["Узкий диапазон", "Растяжение p1–p99"],
    cmaps=["gray", "gray"], vmin=0, vmax=1,
)


### 1.1. Кусочно-линейное преобразование

Произвольную монотонную функцию преобразования удобно задавать опорными точками.
Например, точки $(0,0)$, $(a,0)$, $(b,1)$, $(1,1)$ подавляют значения ниже $a$,
растягивают $[a,b]$ и насыщают значения выше $b$.

**Задание 2.** Реализуйте функцию через `np.interp`, примените её к изображению и
постройте график $s=T(r)$. Сравните результат с перцентильным растяжением.


In [ ]:
def piecewise_contrast(image_float, a, b):
    # TODO: проверьте 0 <= a < b <= 1 и интерполируйте по четырём опорным точкам.
    pass


ramp = np.linspace(0, 1, 256, dtype=np.float32)
a, b = 0.35, 0.65
curve = piecewise_contrast(ramp, a, b)
piecewise_result = piecewise_contrast(to_float(low_contrast), a, b)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(ramp, curve)
axes[0].plot([0, 1], [0, 1], "--", color="gray")
axes[0].set(xlabel="r", ylabel="s", title="Функция преобразования", xlim=(0, 1), ylim=(0, 1))
axes[0].grid(alpha=0.3)
axes[1].imshow(piecewise_result, cmap="gray", vmin=0, vmax=1)
axes[1].set_title("Кусочно-линейный контраст")
axes[1].axis("off")
plt.tight_layout()
plt.show()


## 2. Логарифм и гамма

Для нормированного входа $r\in[0,1]$:

- гамма: $s=r^\gamma$ — настраиваемая сила преобразования;
- логарифм: $s=\log(1+\alpha r)/\log(1+\alpha)$ — сильное сжатие большого диапазона.

### 🔮 Предскажите
Какая гамма осветлит тени: `0.5` или `2.0`? Что произойдёт с шумом в тенях?


In [ ]:
dark = np.clip(to_float(gray) ** 2.2, 0, 1)

# TODO: примените гаммы 0.5 и 2.0.
gamma_half = ...
gamma_two = ...

# TODO: реализуйте нормированный логарифм при alpha=100:
# log(1 + alpha*r) / log(1 + alpha)
alpha = 100.0
log_result = ...

show(
    dark, gamma_half, gamma_two, log_result,
    titles=["Тёмный вход", "γ = 0.5", "γ = 2.0", "Логарифм"],
    cmaps=["gray"] * 4, figsize=(18, 4), vmin=0, vmax=1,
)


## 3. Гистограмма, CDF и эквализация своими руками

Нормированная гистограмма $p(k)=h(k)/N$, а CDF:

$$F(k)=\sum_{j=0}^{k}p(j).$$

Дискретная эквализация использует таблицу замены $T(k)\approx255F(k)$.


In [ ]:
# TODO:
# 1. Посчитайте hist через np.bincount (гарантируйте 256 уровней).
# 2. Нормируйте hist до pdf.
# 3. Найдите cdf накопительной суммой.
# 4. Постройте uint8 LUT: round(255*cdf).
# 5. Примените LUT индексированием к изображению.
hist = ...
pdf = ...
cdf = ...
lut = ...
equalized_manual = ...

equalized_cv = cv2.equalizeHist(low_contrast)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].bar(np.arange(256), hist, width=1)
axes[0].set(title="Гистограмма", xlabel="Яркость", ylabel="Число пикселей")
axes[1].plot(cdf)
axes[1].set(title="CDF", xlabel="Яркость", ylabel="Доля ≤ k", ylim=(0, 1.02))
plt.tight_layout()
plt.show()

# TODO: посчитайте MAE ручной реализации и cv2.equalizeHist.
print("MAE с cv2.equalizeHist:", ...)
show(
    low_contrast, equalized_manual,
    titles=["До", "Ручная эквализация"],
    cmaps=["gray", "gray"], vmin=0, vmax=255,
)


### 3.1. Почему в эквализации вычитают первую ненулевую CDF

Формула $255F(k)$ не обязательно переводит минимальный присутствующий уровень в 0.
Часто используют

$$T(k)=\operatorname{round}\left(255\frac{F(k)-F_{\min}}{1-F_{\min}}\right).$$

**Задание 3.** Реализуйте этот вариант, корректно обработайте постоянное изображение
и сравните MAE с `cv2.equalizeHist`.


In [ ]:
def equalize_with_cdf_min(image):
    # TODO: hist -> CDF в отсчётах -> первая ненулевая CDF -> LUT -> результат.
    # Для постоянного изображения верните копию входа.
    pass


equalized_cdf_min = equalize_with_cdf_min(low_contrast)
mae_cdf_min = np.mean(np.abs(equalized_cdf_min.astype(float) - equalized_cv.astype(float)))
print("MAE после учёта CDF_min:", mae_cdf_min)


### 3.2. Сопоставление гистограмм

Эквализация задаёт целевое равномерное распределение. В более общем случае можно
потребовать, чтобы CDF результата была похожа на CDF эталонного изображения:
$T(r)=F_{ref}^{-1}(F_{src}(r))$.

**Задание 4.** Реализуйте сопоставление для двух `uint8`-изображений через две CDF
и `np.interp`. Сравните с `skimage.exposure.match_histograms`.


In [ ]:
def match_histogram_uint8(source, reference):
    # TODO: постройте CDF source/reference и LUT через обратное отображение CDF.
    # Для обратной интерполяции оставьте только уровни reference с ненулевой частотой.
    pass


reference = data.moon()
matched = match_histogram_uint8(gray, reference)
matched_sk = exposure.match_histograms(gray, reference)

show(
    gray, reference, matched,
    titles=["Источник", "Эталон", "Сопоставленная гистограмма"],
    cmaps=["gray"] * 3, vmin=0, vmax=255,
)
print("MAE с skimage:", np.mean(np.abs(matched.astype(float) - matched_sk)))


## 4. Глобальная эквализация и CLAHE

Глобальная функция преобразования одна для всего изображения. CLAHE строит локальные
преобразования по плиткам и ограничивает усиление гистограммы параметром `clipLimit`.


In [ ]:
# Создадим неоднородное освещение.
x = np.linspace(0.25, 1.0, gray.shape[1], dtype=np.float32)
uneven = np.clip(gray.astype(np.float32) * x[None, :], 0, 255).astype(np.uint8)

# TODO: примените глобальную эквализацию.
global_eq = ...

# TODO: создайте CLAHE с clipLimit=2.0 и сеткой 8x8, затем примените.
clahe = ...
local_eq = ...

show(
    uneven, global_eq, local_eq,
    titles=["Неоднородный свет", "Глобально", "CLAHE"],
    cmaps=["gray"] * 3, vmin=0, vmax=255,
)


### 4.1. Исследование параметров CLAHE

`clipLimit` управляет ограничением усиления, а `tileGridSize` — пространственным
масштабом адаптации. Большой численный разброс яркостей ещё не гарантирует хороший
визуальный результат.

**Задание 5.** Переберите `clipLimit` из `{1, 2, 5, 20}` при сетке `8×8`.
Для каждого результата вычислите `std` и долю пикселей со значениями 0 или 255.
Визуально выберите разумный вариант и объясните, почему максимум `std` не обязан
быть лучшим.


In [ ]:
clahe_results = {}
for clip_limit in [1.0, 2.0, 5.0, 20.0]:
    # TODO: примените CLAHE, сохраните изображение и напечатайте std и долю насыщения.
    pass

# TODO: покажите четыре результата в одной строке с подписями параметров.


## 5. Цветовая сегментация в HSV

Тон `H` описывает тип цвета, но ненадёжен при малых насыщенности `S` и яркости `V`.
В OpenCV для `uint8`: $H\in[0,179]$, а не $[0,360)$.

Выделим синие участки на изображении `astronaut`.


In [ ]:
# TODO:
# 1. Преобразуйте rgb из RGB в HSV средствами OpenCV.
# 2. Разделите H, S, V.
# 3. Создайте маску синего: H в [90,130], S>70, V>40.
# 4. Оставьте исходные цвета только под маской.
hsv = ...
h, s, v = ...
mask_blue = ...
segmented = ...

show(
    rgb, h, s, mask_blue, segmented,
    titles=["RGB", "H", "S", "Маска", "Результат"],
    cmaps=[None, "hsv", "gray", "gray", None], figsize=(20, 4),
)
print("Доля выделенных пикселей:", mask_blue.mean())


### 5.1. Цикличность Hue: красный занимает два диапазона

Красный расположен около границы цветового круга: в шкале OpenCV это значения
около 0 и около 179. Один обычный интервал не охватит обе части.

**Задание 6.** Создайте HSV-полоску со всеми значениями `H`, выделите красный
условием `H <= 10 OR H >= 170` и объясните, почему нужен оператор `OR`.


In [ ]:
hue_strip_hsv = np.zeros((40, 180, 3), dtype=np.uint8)
hue_strip_hsv[..., 0] = np.arange(180, dtype=np.uint8)[None, :]
hue_strip_hsv[..., 1:] = 255
hue_strip_rgb = cv2.cvtColor(hue_strip_hsv, cv2.COLOR_HSV2RGB)

# TODO: постройте булеву маску красного для двух краёв шкалы Hue.
mask_red = ...
show(hue_strip_rgb, mask_red, titles=["Шкала Hue", "Красный: два диапазона"], cmaps=[None, "gray"])


## 6. Баланс белого: Gray World

Гипотеза Gray World: средний цвет достаточно разнообразной сцены должен быть серым.
Оцениваем три коэффициента по средним каналов и применяем их в линейном RGB.
Здесь используем приближённое декодирование/кодирование степенью `2.2`.


In [ ]:
def gray_world(image_rgb):
    # TODO:
    # 1. Нормируйте RGB и декодируйте в линейный RGB степенью 2.2.
    # 2. Найдите среднее каждого канала.
    # 3. Целевое среднее — среднее трёх средних.
    # 4. Вычислите gains = target / channel_means.
    # 5. Примените gains, clip, кодирование степенью 1/2.2 и uint8.
    ...
    return corrected, gains


linear = to_float(rgb) ** 2.2
warm_linear = np.clip(linear * np.array([1.35, 1.0, 0.65], dtype=np.float32), 0, 1)
warm = np.clip(warm_linear ** (1 / 2.2) * 255, 0, 255).astype(np.uint8)

corrected, gains = gray_world(warm)
print("Оценённые коэффициенты:", gains)
show(rgb, warm, corrected, titles=["Исходник", "Тёплый свет", "Gray World"])


### 6.1. Когда Gray World ошибается

Если в сцене закономерно доминирует один цвет, серое среднее — неверная гипотеза.

**Задание 7.** Примените `gray_world` к `data.chelsea()`. Сравните средние каналов
до и после коррекции, визуализируйте результат и объясните, почему выравнивание
средних может ухудшить естественный цвет сцены.


In [ ]:
cat = data.chelsea()
# TODO: примените gray_world, выведите средние RGB до/после и gains, покажите изображения.


## 7. Метод Отсу своими руками

Порог делит гистограмму на два класса. Отсу максимизирует межклассовую дисперсию:

$$\sigma_b^2(t)=\omega_0(t)\omega_1(t)(\mu_0(t)-\mu_1(t))^2.$$

Пустые классы нужно исключить из перебора.


In [ ]:
def otsu_threshold(image):
    hist = np.bincount(image.ravel(), minlength=256).astype(np.float64)
    p = hist / hist.sum()
    levels = np.arange(256)

    # TODO:
    # 1. Накопительно найдите omega0 и первый момент moment0.
    # 2. Для непустых классов найдите mu0 и mu1.
    # 3. Посчитайте sigma_b2 для каждого допустимого порога.
    # 4. Верните argmax и весь массив критерия.
    ...
    return threshold, sigma_b2


coins = data.coins()
t_manual, criterion = otsu_threshold(coins)
t_skimage = filters.threshold_otsu(coins)
mask = coins > t_manual

print("Наш порог:", t_manual, "| skimage:", t_skimage)
show(coins, mask, titles=["coins", f"Отсу, t={t_manual}"], cmaps=["gray", "gray"])

plt.figure(figsize=(9, 3))
plt.plot(criterion)
plt.axvline(t_manual, color="red", linestyle="--")
plt.title("Межклассовая дисперсия")
plt.xlabel("Порог t")
plt.show()


## 8. Документ с тенью: Отсу против Sauvola

Один глобальный порог не может одновременно обработать светлую и затенённую части.
Sauvola использует локальные среднее и стандартное отклонение:

$$t(x,y)=m(x,y)\left[1+k\left(\frac{s(x,y)}{R}-1\right)\right].$$


In [ ]:
page = data.page()
gradient = np.linspace(1.0, 0.35, page.shape[1], dtype=np.float32)
shadowed = np.clip(page.astype(np.float32) * gradient[None, :], 0, 255).astype(np.uint8)

# TODO:
# 1. Найдите глобальный порог Отсу и бинарную маску светлого фона.
# 2. Найдите карту порогов Sauvola: window_size=31, k=0.25, r=128.
# 3. Постройте локальную бинарную маску.
t_global = ...
mask_global = ...
t_local = ...
mask_local = ...

show(
    shadowed, mask_global, t_local, mask_local,
    titles=["Документ с тенью", f"Отсу, t={t_global}", "Порог Sauvola", "Sauvola"],
    cmaps=["gray"] * 4, figsize=(18, 5),
)


### 8.1. Количественное сравнение порогов

При известной эталонной маске сегментацию можно оценить мерой Intersection over Union:

$$IoU=\frac{|A\cap B|}{|A\cup B|}.$$

**Задание 8.** Создайте синтетический тёмный объект на фоне с градиентом яркости,
добавьте гауссов шум и сравните IoU для Отсу и Sauvola. Зафиксируйте seed.


In [ ]:
def binary_iou(mask_a, mask_b):
    # TODO: используйте logical_and / logical_or; обработайте пустое объединение.
    pass


rng = np.random.default_rng(42)
yy, xx = np.mgrid[:256, :256]
truth_dark = (xx - 128) ** 2 + (yy - 128) ** 2 < 55 ** 2
background = 210 - 80 * (xx / 255)
synthetic = background.copy()
synthetic[truth_dark] -= 85
synthetic = np.clip(synthetic + rng.normal(0, 8, synthetic.shape), 0, 255).astype(np.uint8)

# TODO: получите маски тёмного объекта через Отсу и Sauvola и напечатайте две IoU.


## 9. Мини-задача на закрепление

Напишите `enhance_and_segment(image_rgb)`, которая:

1. выполняет Gray World;
2. переводит результат в grayscale;
3. применяет CLAHE (`clipLimit=2.0`, сетка `8×8`);
4. находит порог Отсу вашей реализацией;
5. возвращает улучшенное grayscale-изображение, бинарную маску и порог.


In [ ]:
def enhance_and_segment(image_rgb):
    """Gray World -> grayscale -> CLAHE -> Otsu."""
    # TODO: соберите полный pipeline из функций и методов выше.
    pass


# После реализации раскомментируйте:
# enhanced, final_mask, final_t = enhance_and_segment(rgb)
# show(
#     rgb, enhanced, final_mask,
#     titles=["Вход", "Gray World → grayscale → CLAHE", f"Отсу, t={final_t}"],
#     cmaps=[None, "gray", "gray"],
# )


## 10. Дополнительные задания

1. **Гамма по целевой медиане.** Подберите `gamma` бинарным поиском так, чтобы медиана
   результата была близка к 0.6. Проверьте на двух изображениях.
2. **Собственная LUT.** Реализуйте негатив, соляризацию или S-кривую как таблицу из
   256 значений и примените только индексированием.
3. **Эквализация цвета.** Сравните независимую эквализацию R/G/B с эквализацией только
   канала яркости в HSV или Lab. Объясните цветовые искажения.
4. **Устойчивый Gray World.** Оцените средние каналов только по пикселям, где ни один
   канал не насыщен. Сравните коэффициенты с базовым методом.
5. **Красный объект.** Настройте два диапазона Hue и ограничения S/V на реальном
   изображении; выведите долю маски и покажите результат.
6. **Карта параметров Sauvola.** Переберите три размера окна и три значения `k`,
   посчитайте IoU на синтетике из раздела 8.1 и визуализируйте таблицу 3×3.



## Что важно вынести с практики

- выбор преобразования начинается с диагностики диапазона, гистограммы и требований задачи;
- точечное преобразование не использует соседей, а наклон его кривой управляет локальным контрастом;
- перцентили устойчивее `min/max`, но насыщают крайние значения;
- гистограмма не хранит координаты пикселей, а совпадение гистограмм не означает совпадения изображений;
- вычитание первой ненулевой CDF использует выходной диапазон полнее;
- CLAHE адаптируется к освещению, но его параметры задают компромисс между деталями и шумом;
- HSV удобен для цветовых правил только с ограничениями по `S`, `V` и учётом цикличности `H`;
- баланс белого опирается на гипотезу о сцене и выполняется в линейном RGB;
- Отсу — глобальный порог для двух классов, Sauvola использует локальный контекст;
- визуальное впечатление полезно дополнять воспроизводимым экспериментом и метрикой, например IoU.
